# 01 — NASA PDS data feasibility and coverage

This notebook decides which Moon and Mars measurements are suitable for the later homogeneous-region and process-discovery pipeline. It inventories collections, samples observational products, inspects their metadata and labels, estimates geographic coverage and cross-property overlap, audits resolution and time information, and exports explicit feasibility decisions.

It does **not** infer processes, cluster locations, or search Earth. Moon and Mars are evaluated separately.

## Scientific rules

1. Metadata records are catalog evidence, not physical measurements.
2. Raw, calibrated, and derived products are kept distinct.
3. Coarse measurements are never presented as high-resolution evidence after interpolation.
4. Missing values are not replaced with zero.
5. Product counts are not geographic coverage; coverage is estimated from valid footprints.
6. Automated decisions are preliminary and preserve the evidence used to make them.
7. A failed API query is recorded as a failure, not silently interpreted as zero available data.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import shutil
import time
import xml.etree.ElementTree as ET
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin, urlparse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)

BASE_URL = 'https://pds.nasa.gov/api/search/1'
OUTPUT_ROOT = Path('/kaggle/working/pds_feasibility') if Path('/kaggle/working').exists() else Path('../outputs/pds_feasibility')
CACHE_DIR = OUTPUT_ROOT / 'cache'
LABEL_DIR = OUTPUT_ROOT / 'labels'
for directory in (OUTPUT_ROOT, CACHE_DIR, LABEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

COLLECTION_LIMIT = 30
PRODUCTS_PER_COLLECTION = 20
DETAILS_PER_PROPERTY = 3
REQUEST_TIMEOUT_SECONDS = 90
REQUEST_RETRIES = 4
REQUEST_PAUSE_SECONDS = 0.15
DOWNLOAD_LABELS = True
MAX_LABEL_BYTES = 5_000_000
MAX_TOTAL_DOWNLOAD_BYTES = 80_000_000
COVERAGE_GRID_DEGREES = 2.0
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
print('Output:', OUTPUT_ROOT.resolve())
print('Free disk (GB):', round(shutil.disk_usage(OUTPUT_ROOT).free / 1e9, 2))
print('GPU is not required for this feasibility notebook.')

## Candidate measurements

These are search specifications, not assumptions that every dataset is usable. Search terms deliberately include instrument names and physical-product descriptions. The notebook records which searches succeed and what the returned collections actually contain.

In [ ]:
TARGET_IDS = {
    'Mars': 'urn:nasa:pds:context:target:planet.mars',
    'Moon': 'urn:nasa:pds:context:target:satellite.earth.moon',
}

PROPERTY_SPECS = [
    {'planet': 'Mars', 'property': 'visible_morphology', 'instrument_ids': ['ctx.mro', 'hirise.mro', 'themis.odyssey'], 'expected_instruments': ['CTX', 'HiRISE', 'THEMIS'], 'physical_quantity': 'reflectance or calibrated image intensity', 'minimum_role': 'required'},
    {'planet': 'Mars', 'property': 'topography', 'instrument_ids': ['mola.mgs', 'hrsc.mex'], 'expected_instruments': ['MOLA', 'HRSC'], 'physical_quantity': 'elevation', 'minimum_role': 'required'},
    {'planet': 'Mars', 'property': 'thermal', 'instrument_ids': ['themis.odyssey'], 'expected_instruments': ['THEMIS'], 'physical_quantity': 'temperature, radiance, or thermal inertia', 'minimum_role': 'supporting'},
    {'planet': 'Mars', 'property': 'mineralogy', 'instrument_ids': ['crism.mro'], 'expected_instruments': ['CRISM'], 'physical_quantity': 'spectral radiance/reflectance or mineral indicator', 'minimum_role': 'supporting'},
    {'planet': 'Mars', 'property': 'subsurface_radar', 'instrument_ids': ['sharad.mro'], 'expected_instruments': ['SHARAD'], 'physical_quantity': 'radar echo power and delay', 'minimum_role': 'supporting'},
    {'planet': 'Mars', 'property': 'atmosphere_dust', 'instrument_ids': ['marci.mro', 'mcs.mro'], 'expected_instruments': ['MARCI', 'MCS'], 'physical_quantity': 'image radiance, atmospheric temperature, or aerosol indicator', 'minimum_role': 'supporting'},
    {'planet': 'Mars', 'property': 'hydrogen_elements', 'instrument_ids': ['hend.odyssey', 'grs.odyssey', 'ns.odyssey'], 'expected_instruments': ['HEND', 'GRS', 'NS'], 'physical_quantity': 'neutron/gamma count or derived abundance', 'minimum_role': 'supporting'},
    {'planet': 'Moon', 'property': 'visible_morphology', 'instrument_ids': ['lroc.lro'], 'expected_instruments': ['LROC'], 'physical_quantity': 'reflectance or calibrated image intensity', 'minimum_role': 'required'},
    {'planet': 'Moon', 'property': 'topography', 'instrument_ids': ['lola.lro'], 'expected_instruments': ['LOLA'], 'physical_quantity': 'elevation and laser reflectance', 'minimum_role': 'required'},
    {'planet': 'Moon', 'property': 'thermal', 'instrument_ids': ['diviner.lro'], 'expected_instruments': ['Diviner'], 'physical_quantity': 'brightness or surface temperature and thermal properties', 'minimum_role': 'required'},
    {'planet': 'Moon', 'property': 'radar', 'instrument_ids': ['mini-rf.lro'], 'expected_instruments': ['Mini-RF'], 'physical_quantity': 'radar backscatter and polarization', 'minimum_role': 'supporting'},
    {'planet': 'Moon', 'property': 'hydrogen', 'instrument_ids': ['lend.lro'], 'expected_instruments': ['LEND'], 'physical_quantity': 'neutron count or derived hydrogen indicator', 'minimum_role': 'supporting'},
    {'planet': 'Moon', 'property': 'ultraviolet', 'instrument_ids': ['lamp.lro'], 'expected_instruments': ['LAMP'], 'physical_quantity': 'ultraviolet radiance or reflectance', 'minimum_role': 'supporting'},
    {'planet': 'Moon', 'property': 'illumination_shadow', 'instrument_ids': ['lola.lro', 'lroc.lro'], 'expected_instruments': ['LOLA', 'LROC'], 'physical_quantity': 'illumination duration, shadow, or terrain-derived geometry', 'minimum_role': 'supporting'},
]

property_specs_df = pd.DataFrame(PROPERTY_SPECS)
display(property_specs_df[['planet', 'property', 'physical_quantity', 'expected_instruments', 'minimum_role']])

## Cached, failure-aware PDS client

In [ ]:
session = requests.Session()
session.headers.update({'User-Agent': 'NASA-Space-Apps-PDS-Feasibility/1.0'})
request_log = []

def stable_cache_path(url, params=None, accept='application/json'):
    identity = json.dumps({'url': url, 'params': params or {}, 'accept': accept}, sort_keys=True)
    return CACHE_DIR / f"{hashlib.sha256(identity.encode()).hexdigest()}.json"

def api_get(path, params=None, accept='application/json', use_cache=True):
    url = path if path.startswith('http') else f"{BASE_URL}/{path.lstrip('/')}"
    cache_path = stable_cache_path(url, params, accept)
    if use_cache and cache_path.exists():
        payload = json.loads(cache_path.read_text(encoding='utf-8'))
        request_log.append({'url': url, 'params': json.dumps(params or {}), 'status': 'cache', 'error': ''})
        return payload

    last_error = None
    for attempt in range(REQUEST_RETRIES):
        try:
            response = session.get(url, params=params, headers={'Accept': accept}, timeout=REQUEST_TIMEOUT_SECONDS)
            response.raise_for_status()
            payload = response.json()
            cache_path.write_text(json.dumps(payload), encoding='utf-8')
            request_log.append({'url': response.url, 'params': json.dumps(params or {}), 'status': response.status_code, 'error': ''})
            time.sleep(REQUEST_PAUSE_SECONDS)
            return payload
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt + 1 < REQUEST_RETRIES:
                time.sleep(2 ** attempt)
    request_log.append({'url': url, 'params': json.dumps(params or {}), 'status': 'failed', 'error': str(last_error)})
    raise RuntimeError(f'PDS request failed: {url}: {last_error}')

def ensure_list(value):
    if value is None:
        return []
    return value if isinstance(value, list) else [value]

def text_value(value):
    if value is None:
        return ''
    if isinstance(value, list):
        return ' | '.join(text_value(item) for item in value)
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

def latest_record(record):
    if isinstance(record, list):
        return record[0] if record else {}
    return record or {}

## Metadata extraction

PDS collections use several dictionaries, so field names are not assumed to be identical. Extraction records both normalized values and the source fields from which they came.

In [ ]:
BOUND_PATTERNS = {
    'west': ['westernmost_longitude', 'west_bounding_coordinate', 'minimum_longitude'],
    'east': ['easternmost_longitude', 'east_bounding_coordinate', 'maximum_longitude'],
    'south': ['southernmost_latitude', 'south_bounding_coordinate', 'minimum_latitude'],
    'north': ['northernmost_latitude', 'north_bounding_coordinate', 'maximum_latitude'],
}
RESOLUTION_PATTERNS = ['pixel_resolution', 'pixel_scale', 'spatial_resolution', 'map_resolution', 'sample_resolution', 'ground_sample_distance']
PROCESSING_PATTERNS = ['processing_level', 'processing_level_id']
UNIT_PATTERNS = ['unit', 'units']

def flattened_properties(record):
    properties = record.get('properties') or {}
    return {str(key): value for key, value in properties.items()}

def first_matching_field(properties, patterns):
    for key, value in properties.items():
        lowered = key.lower()
        if any(pattern in lowered for pattern in patterns):
            values = ensure_list(value)
            if values and values[0] not in (None, ''):
                return values[0], key
    return None, ''

def numeric_value(value):
    if value is None:
        return np.nan
    if isinstance(value, dict):
        value = value.get('value') or value.get('_value')
    match = re.search(r'[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?', str(value))
    return float(match.group()) if match else np.nan

def normalize_longitude(value):
    if not np.isfinite(value):
        return np.nan
    return ((float(value) + 180.0) % 360.0) - 180.0

def extract_bounds(properties):
    result = {}
    fields = {}
    for side, patterns in BOUND_PATTERNS.items():
        value, field = first_matching_field(properties, patterns)
        result[side] = numeric_value(value)
        fields[side] = field
    result['west'] = normalize_longitude(result['west'])
    result['east'] = normalize_longitude(result['east'])
    result['bounds_fields'] = json.dumps(fields)
    valid = all(np.isfinite(result[side]) for side in ('west', 'east', 'south', 'north'))
    result['has_bounds'] = bool(valid and -90 <= result['south'] <= result['north'] <= 90)
    return result

def referenced_ids(record, key):
    values = record.get(key) or []
    if isinstance(values, dict):
        values = [values]
    ids = []
    for value in values:
        if isinstance(value, dict):
            ids.append(str(value.get('id') or value.get('title') or ''))
        else:
            ids.append(str(value))
    return ' | '.join(item for item in ids if item)

def normalize_product(record, planet, property_name, collection_id='', search_term=''):
    properties = flattened_properties(record)
    bounds = extract_bounds(properties)
    resolution, resolution_field = first_matching_field(properties, RESOLUTION_PATTERNS)
    processing_level, processing_field = first_matching_field(properties, PROCESSING_PATTERNS)
    unit, unit_field = first_matching_field(properties, UNIT_PATTERNS)
    collection_type, collection_type_field = first_matching_field(properties, ['collection_type'])
    metadata = record.get('metadata') or {}
    row = {
        'planet': planet, 'property': property_name, 'search_term': search_term,
        'collection_id': collection_id, 'product_id': text_value(record.get('id')),
        'product_type': text_value(record.get('type')), 'title': text_value(record.get('title')),
        'description': text_value(record.get('description')),
        'start_date_time': text_value(record.get('start_date_time')),
        'stop_date_time': text_value(record.get('stop_date_time')),
        'instruments': referenced_ids(record, 'observing_system_components'),
        'targets': referenced_ids(record, 'targets'),
        'label_url': text_value(metadata.get('label_url')),
        'version': text_value(metadata.get('version')),
        'processing_level': text_value(processing_level), 'processing_field': processing_field,
        'collection_type': text_value(collection_type), 'collection_type_field': collection_type_field,
        'resolution_value': text_value(resolution), 'resolution_field': resolution_field,
        'unit_value': text_value(unit), 'unit_field': unit_field,
        'property_field_count': len(properties),
        'property_fields': json.dumps(sorted(properties.keys())),
    }
    row.update(bounds)
    return row

## Discover candidate collections

In [ ]:
collection_rows = []
search_rows = []

for spec in PROPERTY_SPECS:
    print(f"{spec['planet']}: {spec['property']}")
    for instrument_suffix in spec['instrument_ids']:
        instrument_id = f'urn:nasa:pds:context:instrument:{instrument_suffix}'
        params = {
            'q': f"((ref_lid_target eq \"{TARGET_IDS[spec['planet']]}\") and (ref_lid_instrument eq \"{instrument_id}\"))",
            'limit': COLLECTION_LIMIT,
        }
        try:
            payload = api_get('classes/collection', params=params)
            records = payload.get('data') or []
            hits = int((payload.get('summary') or {}).get('hits') or len(records))
            search_rows.append({'planet': spec['planet'], 'property': spec['property'], 'search_term': instrument_id, 'hits': hits, 'returned': len(records), 'status': 'ok', 'error': ''})
            for record in records:
                collection_rows.append(normalize_product(record, spec['planet'], spec['property'], search_term=instrument_id))
        except RuntimeError as exc:
            search_rows.append({'planet': spec['planet'], 'property': spec['property'], 'search_term': instrument_id, 'hits': 0, 'returned': 0, 'status': 'failed', 'error': str(exc)})

collection_search_df = pd.DataFrame(search_rows)
collections_df = pd.DataFrame(collection_rows)
if not collections_df.empty:
    collections_df = collections_df.drop_duplicates(['planet', 'property', 'product_id']).reset_index(drop=True)

display(collection_search_df)
print('Unique property-collection matches:', len(collections_df))
if not collections_df.empty:
    display(collections_df[['planet', 'property', 'product_id', 'title', 'processing_level', 'instruments']].head(40))

## Select and sample collections

Search can return broad or irrelevant collections. Selection favors records whose titles, descriptions, identifiers, or instrument references contain the expected instrument names. Up to three collections per property are sampled, and every selection score remains visible.

In [ ]:
def instrument_match_score(row, expected):
    searchable = ' '.join(str(row.get(column, '')) for column in ['product_id', 'title', 'description', 'instruments']).lower()
    return sum(1 for instrument in expected if instrument.lower() in searchable)

selected_parts = []
if not collections_df.empty:
    for spec in PROPERTY_SPECS:
        subset = collections_df[(collections_df['planet'] == spec['planet']) & (collections_df['property'] == spec['property'])].copy()
        if subset.empty:
            continue
        subset['instrument_match_score'] = subset.apply(lambda row: instrument_match_score(row, spec['expected_instruments']), axis=1)
        subset['data_collection_bonus'] = subset['collection_type'].str.lower().eq('data').astype(int) * 100
        subset['document_penalty'] = subset['title'].str.contains('document|calibration|browse', case=False, regex=True, na=False).astype(int) * 80
        subset['metadata_score'] = subset['property_field_count'] + 25 * subset['instrument_match_score'] + subset['data_collection_bonus'] - subset['document_penalty']
        selected_parts.append(subset.sort_values(['instrument_match_score', 'metadata_score'], ascending=False).head(2))

selected_collections_df = pd.concat(selected_parts, ignore_index=True) if selected_parts else pd.DataFrame()
display(selected_collections_df[['planet', 'property', 'product_id', 'title', 'instrument_match_score', 'metadata_score']] if not selected_collections_df.empty else selected_collections_df)

product_rows = []
member_query_rows = []
for _, collection in selected_collections_df.iterrows():
    collection_id = collection['product_id']
    print('Sampling:', collection_id)
    try:
        payload = api_get(f'products/{collection_id}/members', params={'limit': PRODUCTS_PER_COLLECTION})
        records = payload.get('data') or []
        member_query_rows.append({'collection_id': collection_id, 'returned': len(records), 'status': 'ok', 'error': ''})
        for record in records:
            product_rows.append(normalize_product(record, collection['planet'], collection['property'], collection_id=collection_id))
    except RuntimeError as exc:
        member_query_rows.append({'collection_id': collection_id, 'returned': 0, 'status': 'failed', 'error': str(exc)})

products_df = pd.DataFrame(product_rows)
if not products_df.empty:
    products_df = products_df.drop_duplicates(['planet', 'property', 'product_id']).reset_index(drop=True)
member_queries_df = pd.DataFrame(member_query_rows)
print('Sampled unique products:', len(products_df))
display(member_queries_df)

## Product-detail and file audit

Detailed PDS4 JSON is inspected for file references and sizes. XML labels are downloaded only when a usable HTTP URL is exposed and strict per-file and total byte limits are satisfied. Scientific data arrays are not bulk-downloaded in this feasibility pass.

In [ ]:
def walk_json(value, path=''):
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f'{path}.{key}' if path else str(key)
            yield from walk_json(child, child_path)
    elif isinstance(value, list):
        for index, child in enumerate(value):
            yield from walk_json(child, f'{path}[{index}]')
    else:
        yield path, value

def extract_file_candidates(detail):
    flattened = list(walk_json(detail))
    refs = [(path, str(value)) for path, value in flattened if path.lower().endswith(('file_ref', 'label_url')) and value]
    sizes = [(path, numeric_value(value)) for path, value in flattened if 'file_size' in path.lower()]
    rows = []
    for path, ref in refs:
        nearest_size = np.nan
        prefix = path.rsplit('.', 1)[0]
        candidates = [size for size_path, size in sizes if size_path.startswith(prefix) and np.isfinite(size)]
        if candidates:
            nearest_size = candidates[0]
        rows.append({'reference_field': path, 'file_reference': ref, 'reported_size_bytes': nearest_size})
    return rows

detail_rows = []
file_rows = []
if not products_df.empty:
    for (planet, property_name), subset in products_df.groupby(['planet', 'property'], sort=True):
        for _, product in subset.head(DETAILS_PER_PROPERTY).iterrows():
            try:
                detail = api_get(f"products/{product['product_id']}", accept='application/vnd.nasa.pds.pds4+json')
                detail_rows.append({'planet': planet, 'property': property_name, 'product_id': product['product_id'], 'status': 'ok', 'error': ''})
                for candidate in extract_file_candidates(detail):
                    candidate.update({'planet': planet, 'property': property_name, 'product_id': product['product_id']})
                    file_rows.append(candidate)
            except RuntimeError as exc:
                detail_rows.append({'planet': planet, 'property': property_name, 'product_id': product['product_id'], 'status': 'failed', 'error': str(exc)})

details_audit_df = pd.DataFrame(detail_rows)
files_df = pd.DataFrame(file_rows)
print('Detailed products inspected:', len(details_audit_df))
print('File references found:', len(files_df))
display(files_df.head(30) if not files_df.empty else files_df)

In [ ]:
download_rows = []
downloaded_total = 0

def safe_filename(product_id, url):
    digest = hashlib.sha256(url.encode()).hexdigest()[:12]
    suffix = Path(urlparse(url).path).suffix.lower() or '.xml'
    base = re.sub(r'[^A-Za-z0-9_.-]+', '_', product_id)[-100:]
    return f'{base}_{digest}{suffix}'

if DOWNLOAD_LABELS and not files_df.empty:
    label_candidates = files_df[files_df['file_reference'].str.match(r'^https?://', na=False)].copy()
    label_candidates = label_candidates[label_candidates['file_reference'].str.lower().str.endswith(('.xml', '.lbl'))]
    for _, candidate in label_candidates.drop_duplicates('file_reference').iterrows():
        url = candidate['file_reference']
        row = {key: candidate[key] for key in ['planet', 'property', 'product_id', 'file_reference', 'reported_size_bytes']}
        try:
            head = session.head(url, allow_redirects=True, timeout=REQUEST_TIMEOUT_SECONDS)
            content_length = numeric_value(head.headers.get('Content-Length'))
            planned_size = content_length if np.isfinite(content_length) else candidate['reported_size_bytes']
            if np.isfinite(planned_size) and planned_size > MAX_LABEL_BYTES:
                row.update({'status': 'skipped_too_large', 'downloaded_bytes': 0, 'local_path': '', 'sha256': '', 'error': ''})
            elif downloaded_total >= MAX_TOTAL_DOWNLOAD_BYTES:
                row.update({'status': 'skipped_total_limit', 'downloaded_bytes': 0, 'local_path': '', 'sha256': '', 'error': ''})
            else:
                response = session.get(url, timeout=REQUEST_TIMEOUT_SECONDS)
                response.raise_for_status()
                content = response.content
                if len(content) > MAX_LABEL_BYTES or downloaded_total + len(content) > MAX_TOTAL_DOWNLOAD_BYTES:
                    row.update({'status': 'skipped_limit_after_get', 'downloaded_bytes': 0, 'local_path': '', 'sha256': '', 'error': ''})
                else:
                    destination = LABEL_DIR / safe_filename(candidate['product_id'], url)
                    destination.write_bytes(content)
                    digest = hashlib.sha256(content).hexdigest()
                    downloaded_total += len(content)
                    row.update({'status': 'downloaded', 'downloaded_bytes': len(content), 'local_path': str(destination), 'sha256': digest, 'error': ''})
        except requests.RequestException as exc:
            row.update({'status': 'failed', 'downloaded_bytes': 0, 'local_path': '', 'sha256': '', 'error': str(exc)})
        download_rows.append(row)

download_manifest_df = pd.DataFrame(download_rows)
print('Downloaded label bytes:', downloaded_total)
display(download_manifest_df.head(30) if not download_manifest_df.empty else download_manifest_df)

## Validate downloaded labels

In [ ]:
label_validation_rows = []
if not download_manifest_df.empty:
    for _, item in download_manifest_df[download_manifest_df['status'] == 'downloaded'].iterrows():
        path = Path(item['local_path'])
        row = {'product_id': item['product_id'], 'local_path': str(path), 'xml_valid': False, 'root_tag': '', 'element_count': 0, 'error': ''}
        try:
            root = ET.parse(path).getroot()
            row.update({'xml_valid': True, 'root_tag': root.tag, 'element_count': sum(1 for _ in root.iter())})
        except (ET.ParseError, OSError) as exc:
            row['error'] = str(exc)
        label_validation_rows.append(row)

label_validation_df = pd.DataFrame(label_validation_rows)
display(label_validation_df.head(30) if not label_validation_df.empty else label_validation_df)

## Coverage and overlap

Coverage is estimated on an area-weighted latitude/longitude occupancy grid. This is a feasibility estimate, not a replacement for precise footprint geometry. Products without valid bounds remain in the inventory but do not contribute to coverage.

In [ ]:
lat_edges = np.arange(-90, 90 + COVERAGE_GRID_DEGREES, COVERAGE_GRID_DEGREES)
lon_edges = np.arange(-180, 180 + COVERAGE_GRID_DEGREES, COVERAGE_GRID_DEGREES)
lat_centers = (lat_edges[:-1] + lat_edges[1:]) / 2
lon_centers = (lon_edges[:-1] + lon_edges[1:]) / 2
cell_weights = np.cos(np.deg2rad(lat_centers))[:, None] * np.ones((len(lat_centers), len(lon_centers)))

def footprint_mask(west, east, south, north):
    lat_mask = (lat_centers >= south) & (lat_centers <= north)
    if west <= east:
        lon_mask = (lon_centers >= west) & (lon_centers <= east)
    else:
        lon_mask = (lon_centers >= west) | (lon_centers <= east)
    return lat_mask[:, None] & lon_mask[None, :]

coverage_masks = {}
coverage_rows = []
for (planet, property_name), subset in products_df.groupby(['planet', 'property']) if not products_df.empty else []:
    valid = subset[subset['has_bounds'] == True]
    mask = np.zeros_like(cell_weights, dtype=bool)
    repeat_count = np.zeros_like(cell_weights, dtype=np.uint16)
    for _, product in valid.iterrows():
        product_mask = footprint_mask(product['west'], product['east'], product['south'], product['north'])
        mask |= product_mask
        repeat_count += product_mask.astype(np.uint16)
    coverage_masks[(planet, property_name)] = mask
    covered_weight = cell_weights[mask].sum()
    coverage_rows.append({
        'planet': planet, 'property': property_name, 'sampled_products': len(subset),
        'products_with_bounds': len(valid),
        'geolocated_fraction': len(valid) / len(subset) if len(subset) else 0,
        'estimated_surface_coverage_fraction': covered_weight / cell_weights.sum(),
        'estimated_repeat_coverage_fraction': cell_weights[repeat_count >= 2].sum() / cell_weights.sum(),
        'max_sample_repeat_count': int(repeat_count.max()),
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

if not coverage_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharex=True)
    for axis, planet in zip(axes, ['Mars', 'Moon']):
        subset = coverage_df[coverage_df['planet'] == planet].sort_values('estimated_surface_coverage_fraction')
        axis.barh(subset['property'], 100 * subset['estimated_surface_coverage_fraction'])
        axis.set_title(f'{planet}: sampled footprint coverage')
        axis.set_xlabel('Estimated surface coverage (%)')
        axis.grid(axis='x', alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
overlap_rows = []
for planet in ['Mars', 'Moon']:
    keys = sorted([key for key in coverage_masks if key[0] == planet])
    for i, (_, property_a) in enumerate(keys):
        for _, property_b in keys[i:]:
            mask_a = coverage_masks[(planet, property_a)]
            mask_b = coverage_masks[(planet, property_b)]
            intersection = mask_a & mask_b
            union = mask_a | mask_b
            overlap_rows.append({
                'planet': planet, 'property_a': property_a, 'property_b': property_b,
                'surface_intersection_fraction': cell_weights[intersection].sum() / cell_weights.sum(),
                'jaccard_overlap': cell_weights[intersection].sum() / cell_weights[union].sum() if union.any() else 0,
            })

overlap_df = pd.DataFrame(overlap_rows)
display(overlap_df.head(80))

## Resolution, time, format, and missingness audit

In [ ]:
audit_rows = []
for spec in PROPERTY_SPECS:
    if products_df.empty:
        subset = pd.DataFrame()
    else:
        subset = products_df[(products_df['planet'] == spec['planet']) & (products_df['property'] == spec['property'])]
    n = len(subset)
    coverage_match = coverage_df[(coverage_df['planet'] == spec['planet']) & (coverage_df['property'] == spec['property'])] if not coverage_df.empty else pd.DataFrame()
    coverage = coverage_match.iloc[0].to_dict() if not coverage_match.empty else {}
    file_subset = files_df[(files_df['planet'] == spec['planet']) & (files_df['property'] == spec['property'])] if not files_df.empty else pd.DataFrame()
    audit_rows.append({
        'planet': spec['planet'], 'property': spec['property'], 'physical_quantity': spec['physical_quantity'],
        'minimum_role': spec['minimum_role'], 'candidate_collections': int(len(selected_collections_df[(selected_collections_df['planet'] == spec['planet']) & (selected_collections_df['property'] == spec['property'])])) if not selected_collections_df.empty else 0,
        'sampled_products': n,
        'geolocated_fraction': float(coverage.get('geolocated_fraction', 0)),
        'sample_coverage_fraction': float(coverage.get('estimated_surface_coverage_fraction', 0)),
        'resolution_metadata_fraction': float(subset['resolution_value'].astype(bool).mean()) if n else 0,
        'unit_metadata_fraction': float(subset['unit_value'].astype(bool).mean()) if n else 0,
        'time_metadata_fraction': float((subset['start_date_time'].astype(bool) | subset['stop_date_time'].astype(bool)).mean()) if n else 0,
        'processing_level_fraction': float(subset['processing_level'].astype(bool).mean()) if n else 0,
        'file_references': len(file_subset),
        'http_file_references': int(file_subset['file_reference'].str.match(r'^https?://', na=False).sum()) if not file_subset.empty else 0,
        'reported_bytes': int(file_subset['reported_size_bytes'].dropna().sum()) if not file_subset.empty else 0,
        'observed_resolution_values': ' | '.join(sorted(set(subset['resolution_value']) - {''}))[:1000] if n else '',
        'observed_processing_levels': ' | '.join(sorted(set(subset['processing_level']) - {''}))[:500] if n else '',
        'observed_units': ' | '.join(sorted(set(subset['unit_value']) - {''}))[:500] if n else '',
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

## Preliminary feasibility decisions

These rules judge data accessibility, not whether a geological interpretation is true. A result can be `Pilot only` because useful data exist but the sampled spatial metadata or coverage is incomplete. `Supporting only` means the property should not independently establish a process.

In [ ]:
def feasibility_decision(row):
    reasons = []
    if row['candidate_collections'] == 0 or row['sampled_products'] == 0:
        return 'Reject', 'No usable collection members were discovered through the tested API route.'
    if row['http_file_references'] == 0:
        reasons.append('No directly downloadable HTTP file reference was found in the detailed sample.')
    if row['geolocated_fraction'] < 0.20:
        reasons.append('Fewer than 20% of sampled products expose parseable bounding coordinates.')
    if row['resolution_metadata_fraction'] < 0.20:
        reasons.append('Resolution metadata are sparse in the sampled API records.')
    if row['time_metadata_fraction'] < 0.20:
        reasons.append('Acquisition-time metadata are sparse, limiting temporal analysis.')

    if row['minimum_role'] == 'supporting':
        decision = 'Supporting only'
        reasons.insert(0, 'This measurement is supporting evidence and cannot establish a process by itself.')
    elif row['http_file_references'] > 0 and row['geolocated_fraction'] >= 0.50 and row['resolution_metadata_fraction'] >= 0.20:
        decision = 'Accept'
        reasons.insert(0, 'The sample exposes products, file references, geolocation, and usable scale metadata.')
    else:
        decision = 'Pilot only'
        reasons.insert(0, 'Products exist, but regional validation or collection-specific access is required before scaling.')
    return decision, ' '.join(reasons)

decision_values = audit_df.apply(feasibility_decision, axis=1)
audit_df['decision'] = [value[0] for value in decision_values]
audit_df['decision_reason'] = [value[1] for value in decision_values]
display(audit_df[['planet', 'property', 'decision', 'decision_reason', 'candidate_collections', 'sampled_products', 'geolocated_fraction', 'sample_coverage_fraction']])

## Compute and storage estimate

In [ ]:
reported_bytes = int(files_df['reported_size_bytes'].dropna().sum()) if not files_df.empty else 0
known_file_count = int(files_df['reported_size_bytes'].notna().sum()) if not files_df.empty else 0
median_file_bytes = float(files_df['reported_size_bytes'].dropna().median()) if known_file_count else np.nan
scaled_product_count = 10_000
estimated_scaled_bytes = int(median_file_bytes * scaled_product_count) if np.isfinite(median_file_bytes) else None
compute_estimate = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'sampled_products': int(len(products_df)),
    'discovered_file_references': int(len(files_df)),
    'file_references_with_reported_size': known_file_count,
    'reported_sample_bytes': reported_bytes,
    'median_reported_file_bytes': None if not np.isfinite(median_file_bytes) else median_file_bytes,
    'example_scaled_product_count': scaled_product_count,
    'estimated_scaled_bytes_from_sample_median': estimated_scaled_bytes,
    'current_free_disk_bytes': shutil.disk_usage(OUTPUT_ROOT).free,
    'warning': 'This extrapolation is preliminary because archive file sizes vary strongly by instrument and product level.'
}
display(pd.Series(compute_estimate, name='value').to_frame())

## Export reproducible artifacts and report

In [ ]:
def safe_parquet(df, path):
    try:
        df.to_parquet(path, index=False)
        return str(path)
    except (ImportError, ValueError, TypeError) as exc:
        csv_path = path.with_suffix('.csv')
        df.to_csv(csv_path, index=False)
        return f'{csv_path} (Parquet unavailable: {exc})'

footprint_rows = []
if not products_df.empty:
    for _, product in products_df[products_df['has_bounds'] == True].iterrows():
        west, east, south, north = product['west'], product['east'], product['south'], product['north']
        if west <= east:
            wkt = f'POLYGON (({west} {south}, {east} {south}, {east} {north}, {west} {north}, {west} {south}))'
        else:
            wkt = f'MULTIPOLYGON ((({west} {south}, 180 {south}, 180 {north}, {west} {north}, {west} {south})), ((-180 {south}, {east} {south}, {east} {north}, -180 {north}, -180 {south})))'
        footprint_rows.append({
            'planet': product['planet'], 'property': product['property'], 'product_id': product['product_id'],
            'west': west, 'east': east, 'south': south, 'north': north, 'geometry_wkt': wkt
        })
footprints_df = pd.DataFrame(footprint_rows)

artifacts = []
artifacts.append(safe_parquet(collections_df, OUTPUT_ROOT / 'catalog_inventory.parquet'))
artifacts.append(safe_parquet(products_df, OUTPUT_ROOT / 'sampled_products.parquet'))
collection_search_df.to_csv(OUTPUT_ROOT / 'collection_search_log.csv', index=False)
selected_collections_df.to_csv(OUTPUT_ROOT / 'collection_audit.csv', index=False)
coverage_df.to_csv(OUTPUT_ROOT / 'property_coverage_summary.csv', index=False)
footprints_df.to_csv(OUTPUT_ROOT / 'product_footprints.csv', index=False)
audit_df[['planet', 'property', 'observed_resolution_values', 'resolution_metadata_fraction']].to_csv(OUTPUT_ROOT / 'resolution_matrix.csv', index=False)
overlap_df.to_csv(OUTPUT_ROOT / 'evidence_overlap_summary.csv', index=False)
audit_df.to_csv(OUTPUT_ROOT / 'feasibility_decisions.csv', index=False)
files_df.to_csv(OUTPUT_ROOT / 'file_inventory.csv', index=False)
download_manifest_df.to_csv(OUTPUT_ROOT / 'download_manifest.csv', index=False)
pd.DataFrame(request_log).to_csv(OUTPUT_ROOT / 'api_request_log.csv', index=False)
(OUTPUT_ROOT / 'compute_storage_estimate.json').write_text(json.dumps(compute_estimate, indent=2), encoding='utf-8')
try:
    import geopandas as gpd
    from shapely import wkt as shapely_wkt
    if not footprints_df.empty:
        footprint_geo = gpd.GeoDataFrame(footprints_df.drop(columns='geometry_wkt'), geometry=footprints_df['geometry_wkt'].map(shapely_wkt.loads), crs='EPSG:4326')
        footprint_geo.to_parquet(OUTPUT_ROOT / 'product_footprints.geoparquet', index=False)
except (ImportError, ValueError, TypeError) as exc:
    print('GeoParquet export unavailable; CSV with WKT was saved:', exc)

report_lines = [
    '# PDS data feasibility report', '',
    f"Generated: {datetime.now(timezone.utc).isoformat()}", '',
    'This report evaluates data accessibility and metadata completeness. It does not validate planetary processes.', ''
]
for planet in ['Mars', 'Moon']:
    report_lines.extend([f'## {planet}', ''])
    subset = audit_df[audit_df['planet'] == planet]
    for _, row in subset.iterrows():
        report_lines.extend([
            f"### {row['property']} — {row['decision']}", '',
            row['decision_reason'], '',
            f"- Candidate collections: {row['candidate_collections']}",
            f"- Sampled products: {row['sampled_products']}",
            f"- Products with usable bounds: {row['geolocated_fraction']:.1%}",
            f"- Estimated sampled footprint coverage: {row['sample_coverage_fraction']:.1%}",
            f"- Direct HTTP file references: {row['http_file_references']}", ''
        ])
report_lines.extend([
    '## Required manual review', '',
    '- Confirm each selected collection against its official Software Interface Specification or collection documentation.',
    '- Verify physical units, quality masks, valid ranges, map projections, and native resolution from downloaded labels.',
    '- Replace sampled bounding-box estimates with collection-specific footprint geometry before global claims.',
    '- Test known positive and negative geographic controls before process discovery.',
    '- Use regional pilots if global multi-property overlap is insufficient.', ''
])
report_path = OUTPUT_ROOT / 'feasibility_report.md'
report_path.write_text('\n'.join(report_lines), encoding='utf-8')

print('Artifacts:')
for path in sorted(OUTPUT_ROOT.iterdir()):
    if path.is_file():
        print(f'  {path.name}: {path.stat().st_size:,} bytes')
display(Markdown(report_path.read_text(encoding='utf-8')))

## Exit gate

Do not start large-scale segmentation or clustering until representative scientific files—not only metadata and labels—have been opened and validated for every accepted property. If global overlap is weak, the process-discovery phase must use several well-covered regional pilots and must not claim global completeness.